# Notebook 3 — Stereo-Aware EGNN (Chirality + E/Z) on QM9

**Project:** Equivariant Graph Neural Networks with Chirality Encoding for Structure-Sensitive Molecular Property Prediction

**Goal of this notebook:**
1. Extract **R/S tetrahedral chirality** and **E/Z double-bond stereochemistry** with RDKit and
   encode them as node/edge features.
2. Add a chirality-sensitive **signed-volume** geometric feature at each stereocentre (a scalar
   triple product that flips sign under mirror reflection — this is the key ingredient a plain
   distance-based EGNN is missing).
3. Feed these into the *same* EGNN message-passing scheme from Notebook 2 — it becomes
   "stereo-aware" simply by being given stereo-sensitive inputs — and retrain on the QM9
   HOMO-LUMO gap task as a sanity check.
4. Build a **stereoisomer discrimination test**: curated R/S and E/Z pairs sharing identical 2D
   connectivity, and measure whether the encoder's embeddings can tell them apart.

**Important scientific caveat (read this before interpreting results):** QM9's scalar targets
(HOMO-LUMO gap, energies, `u0`, etc.) are **physically identical for a pair of enantiomers** —
mirror symmetry doesn't change a molecule's energy spectrum in isolation. So we should *not*
expect (or want) the model to predict different `gap` values for R vs S. What this notebook
actually demonstrates is a **representation-level** capability: can the encoder produce
*different embeddings* for stereoisomers that share identical 2D graphs? That capability is a
prerequisite for later notebooks (Tox21 / BBBP), where enantiomers genuinely can have very
different biological activity (the classic textbook example being thalidomide).

**Before running this notebook:** upload `qm9_split.pt` and `target_norm_stats.pt` from
Notebook 1.


In [ ]:
!pip install -q torch_geometric
!pip install -q rdkit


In [ ]:
import os
import json
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader as TorchDataLoader
from torch_geometric.datasets import QM9
from torch_geometric.data import Data, Batch
from torch_geometric.nn import global_mean_pool
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)


## 1. Load QM9 and reuse the same split + subsample as Notebook 2

We use the exact same molecules as Notebook 2's 3D EGNN so the two test MAEs are directly
comparable.


In [ ]:
DATA_ROOT = './data/QM9'
dataset = QM9(root=DATA_ROOT)

SPLIT_PATH = 'qm9_split.pt'
NORM_PATH = 'target_norm_stats.pt'

TARGET_NAMES = [
    'mu', 'alpha', 'homo', 'lumo', 'gap', 'r2', 'zpve', 'u0', 'u298',
    'h298', 'g298', 'cv', 'u0_atom', 'u298_atom', 'h298_atom',
    'g298_atom', 'A', 'B', 'C'
]

if os.path.exists(SPLIT_PATH):
    split = torch.load(SPLIT_PATH)
    train_idx, val_idx, test_idx = split['train_idx'], split['val_idx'], split['test_idx']
    TARGET_IDX = split['target_idx']
    print("Loaded split from Notebook 1.")
else:
    print("qm9_split.pt not found — regenerating an equivalent split with the same seed/logic.")
    TARGET_IDX = 4
    NUM_SAMPLES = 20000
    perm = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(SEED))
    subset_indices = perm[:NUM_SAMPLES]
    n_train = int(0.8 * NUM_SAMPLES)
    n_val = int(0.1 * NUM_SAMPLES)
    train_idx = subset_indices[:n_train]
    val_idx = subset_indices[n_train:n_train + n_val]
    test_idx = subset_indices[n_train + n_val:]

print(f"Predicting target: {TARGET_NAMES[TARGET_IDX]} (index {TARGET_IDX})")

if os.path.exists(NORM_PATH):
    norm_stats = torch.load(NORM_PATH)
    target_mean, target_std = norm_stats['mean'], norm_stats['std']
else:
    train_targets = torch.stack([dataset[i].y[0, TARGET_IDX] for i in train_idx])
    target_mean, target_std = train_targets.mean(), train_targets.std()

print(f"Target mean: {target_mean:.4f}, std: {target_std:.4f}")

# Same subsample sizes/seed as Notebook 2, so this is the same set of molecules.
MAX_TRAIN, MAX_VAL, MAX_TEST = 3000, 500, 500
g = torch.Generator().manual_seed(SEED)
train_idx_3d = train_idx[torch.randperm(len(train_idx), generator=g)[:MAX_TRAIN]]
val_idx_3d = val_idx[torch.randperm(len(val_idx), generator=g)[:MAX_VAL]]
test_idx_3d = test_idx[torch.randperm(len(test_idx), generator=g)[:MAX_TEST]]

print(f"3D subset -> Train: {len(train_idx_3d)}, Val: {len(val_idx_3d)}, Test: {len(test_idx_3d)}")


## 2. Conformer generation (same ETKDGv3 + MMFF94 pipeline as Notebook 2)

One difference from Notebook 2: this version returns the RDKit `Mol` object together with the
**conformer IDs** of the selected low-energy conformers (rather than raw coordinate arrays), so
we can run `Chem.AssignStereochemistryFrom3D` directly on each specific conformer without any
extra bookkeeping.


In [ ]:
N_CONFORMERS = 3


def generate_conformers(smiles, num_confs=N_CONFORMERS, num_attempts=20):
    """Embed `num_confs` low-energy conformers for `smiles`.
    Returns (mol, best_conf_ids) or None on failure. `mol` retains ALL
    generated conformers; `best_conf_ids` picks out the lowest-MMFF-energy
    ones (a proxy for thermodynamically representative geometries).
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = SEED
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=max(num_confs, num_attempts), params=params)
    if len(conf_ids) == 0:
        return None

    energies = []
    for cid in conf_ids:
        try:
            props = AllChem.MMFFGetMoleculeProperties(mol)
            ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            ff.Minimize()
            energies.append((cid, ff.CalcEnergy()))
        except Exception:
            energies.append((cid, float('inf')))

    energies.sort(key=lambda t: t[1])
    best_conf_ids = [cid for cid, e in energies[:num_confs] if np.isfinite(e)]
    if len(best_conf_ids) == 0:
        return None

    while len(best_conf_ids) < num_confs:
        best_conf_ids.append(best_conf_ids[-1])

    return mol, best_conf_ids


## 3. Stereo feature extraction

- **R/S**: after `Chem.AssignStereochemistryFrom3D` (which derives chiral tags from the actual
  3D geometry of a specific conformer) + `Chem.AssignStereochemistry(cleanIt=True, force=True)`
  (which fills in CIP `_CIPCode` atom properties), each stereocentre atom gets `'R'`, `'S'`, or
  no tag.
- **E/Z**: `bond.GetStereo()` returns `STEREOE` / `STEREOZ` / none for double bonds, using the
  same CIP-based assignment.
- **Signed volume** (the key chirality-sensitive *geometric* feature): for a stereocentre atom,
  take vectors to its first three neighbours (sorted by atom index, for a deterministic
  convention) and compute the scalar triple product. This quantity is invariant under rotation
  and translation but **flips sign under reflection** — exactly the property a plain
  distance-based EGNN lacks, since squared distances alone cannot distinguish a structure from
  its mirror image.


In [ ]:
def compute_signed_volume(mol, atom_idx, conf):
    atom = mol.GetAtomWithIdx(atom_idx)
    neighbor_idxs = sorted(n.GetIdx() for n in atom.GetNeighbors())
    if len(neighbor_idxs) < 3:
        return 0.0
    p0 = conf.GetAtomPosition(atom_idx)
    p0 = np.array([p0.x, p0.y, p0.z])
    pts = []
    for n_idx in neighbor_idxs[:3]:
        p = conf.GetAtomPosition(n_idx)
        pts.append(np.array([p.x, p.y, p.z]) - p0)
    return float(np.dot(pts[0], np.cross(pts[1], pts[2])))


## 4. Build PyG `Data` objects with stereo features

Node features: atom-type one-hot (5) + chirality one-hot (`none`/`R`/`S`, 3) + signed-volume
scalar (1) = **9 dims**. Edge features: bond-type one-hot (4) + bond-stereo one-hot
(`none`/`E`/`Z`, 3) = **7 dims**. We also keep the stereo-*blind* builder from Notebook 2
around (renamed `mol_to_data_blind`) — we'll need it later for the discrimination test.


In [ ]:
ATOM_TYPES = ['H', 'C', 'N', 'O', 'F']
ATOM_TO_IDX = {a: i for i, a in enumerate(ATOM_TYPES)}
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE, Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]

CHIRALITY_TYPES = ['none', 'R', 'S']
CHIRALITY_TO_IDX = {c: i for i, c in enumerate(CHIRALITY_TYPES)}
STEREO_BOND_TYPES = ['none', 'E', 'Z']
STEREO_BOND_TO_IDX = {c: i for i, c in enumerate(STEREO_BOND_TYPES)}


def mol_to_data_stereo(mol, conf_id, y_value):
    """Stereo-aware graph builder: atom type + chirality tag + signed
    volume on nodes; bond type + E/Z tag on edges."""
    Chem.AssignStereochemistryFrom3D(mol, confId=conf_id)
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
    conf = mol.GetConformer(conf_id)

    atoms = mol.GetAtoms()
    x = torch.zeros(len(atoms), len(ATOM_TYPES) + len(CHIRALITY_TYPES) + 1)
    for atom in atoms:
        i = atom.GetIdx()
        a_idx = ATOM_TO_IDX.get(atom.GetSymbol())
        if a_idx is None:
            return None
        x[i, a_idx] = 1.0

        cip = atom.GetProp('_CIPCode') if atom.HasProp('_CIPCode') else None
        x[i, len(ATOM_TYPES) + CHIRALITY_TO_IDX.get(cip, 0)] = 1.0
        if cip is not None:
            x[i, -1] = compute_signed_volume(mol, i, conf)

    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = bond.GetBondType()
        bt_onehot = [1.0 if bt == b else 0.0 for b in BOND_TYPES]

        st = bond.GetStereo()
        if st == Chem.BondStereo.STEREOE:
            s_label = 'E'
        elif st == Chem.BondStereo.STEREOZ:
            s_label = 'Z'
        else:
            s_label = 'none'
        s_onehot = [0.0, 0.0, 0.0]
        s_onehot[STEREO_BOND_TO_IDX[s_label]] = 1.0

        full_attr = bt_onehot + s_onehot
        edge_index += [[i, j], [j, i]]
        edge_attr += [full_attr, full_attr]

    if len(edge_index) == 0:
        return None

    coords = conf.GetPositions()
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    pos = torch.tensor(coords, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, pos=pos,
                y=torch.tensor([y_value], dtype=torch.float))


def mol_to_data_blind(mol, conf_id, y_value):
    """Stereo-BLIND graph builder (identical to Notebook 2): atom type
    only on nodes, bond type only on edges. Used later purely as a
    reference point for the discrimination test."""
    conf = mol.GetConformer(conf_id)
    atoms = mol.GetAtoms()
    x = torch.zeros(len(atoms), len(ATOM_TYPES))
    for atom in atoms:
        idx = ATOM_TO_IDX.get(atom.GetSymbol())
        if idx is None:
            return None
        x[atom.GetIdx(), idx] = 1.0

    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = bond.GetBondType()
        bt_onehot = [1.0 if bt == b else 0.0 for b in BOND_TYPES]
        edge_index += [[i, j], [j, i]]
        edge_attr += [bt_onehot, bt_onehot]

    if len(edge_index) == 0:
        return None

    coords = conf.GetPositions()
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    pos = torch.tensor(coords, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, pos=pos,
                y=torch.tensor([y_value], dtype=torch.float))


## 5. Build (and cache) the stereo-aware conformer datasets

Same idea as Notebook 2: slow to build, so cache to disk under new filenames (these are
different `Data` objects — wider node/edge features — so they must not be confused with
Notebook 2's `conformers_*.pkl` cache).


In [ ]:
def build_conformer_dataset(indices, target_idx, mean, std, cache_path):
    if os.path.exists(cache_path):
        print(f"Loading cached conformer dataset from {cache_path}")
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    samples = []
    skipped = 0
    for idx in tqdm(indices.tolist(), desc=f"Generating conformers -> {cache_path}"):
        data = dataset[idx]
        smiles = getattr(data, 'smiles', None)
        if smiles is None:
            skipped += 1
            continue

        result = generate_conformers(smiles)
        if result is None:
            skipped += 1
            continue
        mol, conf_ids = result

        y_raw = data.y[0, target_idx].item()
        y_norm = (y_raw - mean.item()) / std.item()

        mol_data_list = []
        ok = True
        for cid in conf_ids:
            d = mol_to_data_stereo(mol, cid, y_norm)
            if d is None:
                ok = False
                break
            mol_data_list.append(d)
        if not ok:
            skipped += 1
            continue

        samples.append(mol_data_list)

    print(f"Built {len(samples)} molecules, skipped {skipped}.")
    with open(cache_path, 'wb') as f:
        pickle.dump(samples, f)
    return samples


train_conf_data = build_conformer_dataset(train_idx_3d, TARGET_IDX, target_mean, target_std, 'conformers_stereo_train.pkl')
val_conf_data = build_conformer_dataset(val_idx_3d, TARGET_IDX, target_mean, target_std, 'conformers_stereo_val.pkl')
test_conf_data = build_conformer_dataset(test_idx_3d, TARGET_IDX, target_mean, target_std, 'conformers_stereo_test.pkl')


## 6. Dataset + collate (identical pattern to Notebook 2)


In [ ]:
class ConformerEnsembleDataset(torch.utils.data.Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        return self.samples[i]


def collate_conformers(batch):
    n_conf = len(batch[0])
    conformer_batches = []
    for k in range(n_conf):
        graphs_k = [mol[k] for mol in batch]
        conformer_batches.append(Batch.from_data_list(graphs_k))
    return conformer_batches


train_loader_3d = TorchDataLoader(ConformerEnsembleDataset(train_conf_data), batch_size=32,
                                   shuffle=True, collate_fn=collate_conformers)
val_loader_3d = TorchDataLoader(ConformerEnsembleDataset(val_conf_data), batch_size=64,
                                 shuffle=False, collate_fn=collate_conformers)
test_loader_3d = TorchDataLoader(ConformerEnsembleDataset(test_conf_data), batch_size=64,
                                  shuffle=False, collate_fn=collate_conformers)


## 7. EGNN layer / encoder / ensemble model

Architecturally **identical** to Notebook 2 — the model becomes "stereo-aware" purely because
its node features now include chirality tags + signed volume, and its edge features include
E/Z tags. Messages `m_ij = phi(h_i, h_j, dist_sq, edge_attr)` therefore naturally become
different for a stereocentre in its `R` vs `S` configuration, since `h_i`/`h_j` already encode
that difference.


In [ ]:
class EGNNLayer(nn.Module):
    def __init__(self, hidden_dim, edge_feat_dim):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1 + edge_feat_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, h, pos, edge_index, edge_attr):
        row, col = edge_index
        rel_pos = pos[row] - pos[col]
        dist_sq = (rel_pos ** 2).sum(dim=-1, keepdim=True)

        edge_input = torch.cat([h[row], h[col], dist_sq, edge_attr], dim=-1)
        m_ij = self.edge_mlp(edge_input)

        coord_weight = self.coord_mlp(m_ij)
        coord_update = rel_pos * coord_weight
        agg_coord = torch.zeros_like(pos)
        agg_coord.index_add_(0, row, coord_update)
        deg = torch.zeros(pos.size(0), 1, device=pos.device)
        deg.index_add_(0, row, torch.ones_like(coord_weight))
        pos_new = pos + agg_coord / deg.clamp(min=1.0)

        agg_m = torch.zeros_like(h)
        agg_m.index_add_(0, row, m_ij)
        h_new = h + self.node_mlp(torch.cat([h, agg_m], dim=-1))

        return h_new, pos_new


class EGNNEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=64, num_layers=4):
        super().__init__()
        self.embed = nn.Linear(node_in_dim, hidden_dim)
        self.layers = nn.ModuleList([EGNNLayer(hidden_dim, edge_in_dim) for _ in range(num_layers)])

    def forward(self, x, pos, edge_index, edge_attr, batch):
        h = self.embed(x)
        for layer in self.layers:
            h, pos = layer(h, pos, edge_index, edge_attr)
        return global_mean_pool(h, batch)


class EGNNConformerEnsemble(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=64, num_layers=4):
        super().__init__()
        self.encoder = EGNNEncoder(node_in_dim, edge_in_dim, hidden_dim, num_layers)
        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, conformer_batches):
        embeddings = []
        for batch_k in conformer_batches:
            emb_k = self.encoder(batch_k.x, batch_k.pos, batch_k.edge_index,
                                  batch_k.edge_attr, batch_k.batch)
            embeddings.append(emb_k)
        ensemble_embedding = torch.stack(embeddings, dim=0).mean(dim=0)
        out = self.readout(ensemble_embedding)
        return out.view(-1)


In [ ]:
node_in_dim = len(ATOM_TYPES) + len(CHIRALITY_TYPES) + 1  # 5 + 3 + 1 = 9
edge_in_dim = len(BOND_TYPES) + len(STEREO_BOND_TYPES)      # 4 + 3 = 7

model_stereo = EGNNConformerEnsemble(node_in_dim, edge_in_dim).to(device)
print(model_stereo)


## 8. Train on the QM9 gap task (sanity check)

We expect test MAE here to be **close to** Notebook 2's 3D EGNN (not necessarily better —
`gap` doesn't depend on chirality, so this mainly checks that the extra stereo features don't
introduce noise or hurt optimization).


In [ ]:
def train_one_epoch_3d(model, loader, optimizer, device):
    model.train()
    total_loss, n = 0.0, 0
    for conformer_batches in loader:
        conformer_batches = [b.to(device) for b in conformer_batches]
        y = conformer_batches[0].y
        optimizer.zero_grad()
        pred = model(conformer_batches)
        loss = F.l1_loss(pred, y)
        loss.backward()
        optimizer.step()
        bs = y.size(0)
        total_loss += loss.item() * bs
        n += bs
    return total_loss / n


@torch.no_grad()
def evaluate_3d(model, loader, device, mean, std):
    model.eval()
    total_mae_norm, total_mae_real, n = 0.0, 0.0, 0
    for conformer_batches in loader:
        conformer_batches = [b.to(device) for b in conformer_batches]
        y = conformer_batches[0].y
        pred = model(conformer_batches)
        total_mae_norm += F.l1_loss(pred, y, reduction='sum').item()
        pred_real = pred * std.to(device) + mean.to(device)
        y_real = y * std.to(device) + mean.to(device)
        total_mae_real += F.l1_loss(pred_real, y_real, reduction='sum').item()
        n += y.size(0)
    return total_mae_norm / n, total_mae_real / n


optimizer_st = torch.optim.Adam(model_stereo.parameters(), lr=1e-3)
scheduler_st = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_st, mode='min', factor=0.5, patience=5)

N_EPOCHS_ST = 60
best_val_mae_st = float('inf')
history_st = {'train_loss': [], 'val_mae_norm': [], 'val_mae_real': []}

for epoch in range(1, N_EPOCHS_ST + 1):
    train_loss = train_one_epoch_3d(model_stereo, train_loader_3d, optimizer_st, device)
    val_mae_norm, val_mae_real = evaluate_3d(model_stereo, val_loader_3d, device, target_mean, target_std)
    scheduler_st.step(val_mae_norm)

    history_st['train_loss'].append(train_loss)
    history_st['val_mae_norm'].append(val_mae_norm)
    history_st['val_mae_real'].append(val_mae_real)

    if val_mae_norm < best_val_mae_st:
        best_val_mae_st = val_mae_norm
        torch.save(model_stereo.state_dict(), 'egnn_stereo_best.pt')

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train L1 (norm): {train_loss:.4f} | "
              f"Val MAE (norm): {val_mae_norm:.4f} | Val MAE (real units): {val_mae_real:.4f}")


In [ ]:
model_stereo.load_state_dict(torch.load('egnn_stereo_best.pt'))
test_mae_norm_st, test_mae_real_st = evaluate_3d(model_stereo, test_loader_3d, device, target_mean, target_std)
print(f"Test MAE (normalized): {test_mae_norm_st:.4f}")
print(f"Test MAE (real units, {TARGET_NAMES[TARGET_IDX]}): {test_mae_real_st:.4f}")

results_stereo = {
    'model_name': '3D_stereo_EGNN',
    'target': TARGET_NAMES[TARGET_IDX],
    'target_idx': TARGET_IDX,
    'num_conformers': N_CONFORMERS,
    'num_train_molecules': len(train_conf_data),
    'num_val_molecules': len(val_conf_data),
    'num_test_molecules': len(test_conf_data),
    'test_mae_normalized': test_mae_norm_st,
    'test_mae_real_units': test_mae_real_st,
}
with open('results_3d_stereo_egnn.json', 'w') as f:
    json.dump(results_stereo, f, indent=2)

if os.path.exists('results_3d_egnn.json'):
    with open('results_3d_egnn.json') as f:
        results_3d = json.load(f)
    print()
    print(f"3D EGNN (no stereo)   test MAE: {results_3d['test_mae_real_units']:.4f}")
    print(f"3D + Stereo EGNN      test MAE: {results_stereo['test_mae_real_units']:.4f}")
    print("(Expected: close to each other -- gap is an achiral property, so stereo features "
          "shouldn't help or hurt much here. This is a sanity check, not the main result.)")


## 9. Rotation-invariance sanity check

Adding chirality/stereo features should **not** break rotation invariance: a proper rotation
(determinant = +1) never changes a molecule's chirality, so predictions should stay unchanged,
just as in Notebook 2.


In [ ]:
@torch.no_grad()
def check_rotation_invariance(model, loader, device, n_batches=1):
    model.eval()
    for b_idx, conformer_batches in enumerate(loader):
        conformer_batches = [b.to(device) for b in conformer_batches]
        pred_original = model(conformer_batches)

        Q, _ = torch.linalg.qr(torch.randn(3, 3, device=device))
        if torch.det(Q) < 0:
            Q[:, 0] *= -1  # force a proper rotation (det = +1), not a reflection
        translation = torch.randn(1, 3, device=device) * 5.0

        rotated = [b.clone() for b in conformer_batches]
        for b in rotated:
            b.pos = b.pos @ Q + translation
        pred_rotated = model(rotated)

        max_diff = (pred_original - pred_rotated).abs().max().item()
        print(f"Batch {b_idx}: max |pred_original - pred_rotated| = {max_diff:.6f} (should be ~0)")

        if b_idx + 1 >= n_batches:
            break


check_rotation_invariance(model_stereo, test_loader_3d, device)


## 10. Stereoisomer discrimination test

This is the core deliverable the project plan asks for: *"test the model on stereoisomer pairs
to check whether it can distinguish molecules that have the same 2D connectivity but different
spatial configurations."*

**Method:**
1. Take small QM9-compatible base molecules (only H, C, N, O, F) written **without** stereo
   descriptors, each containing exactly one stereocentre or one stereogenic double bond.
2. Use RDKit's `EnumerateStereoisomers` to generate the two distinct stereoisomers (e.g. R and
   S, or E and Z).
3. Embed one low-energy 3D conformer for each isomer.
4. Compare embeddings from **two encoders**:
   - A **stereo-blind** `EGNNEncoder` (Notebook 2's architecture, 5-dim node / 4-dim edge
     features). We use it **freshly initialized, untrained** — and this is intentional, not a
     shortcut: since its only geometric input is squared distance (invariant under rotation
     *and* reflection), **any** set of weights gives identical output for a molecule and its
     mirror image. The invariance is architectural, not learned, so training doesn't matter for
     this particular check.
   - The **trained stereo-aware encoder** from this notebook (`model_stereo.encoder`).

We report L2 distance and cosine similarity between the two isomers' embeddings under each
encoder. Expected result: near-zero difference under the blind encoder, clearly nonzero
difference under the stereo-aware encoder.


In [ ]:
STEREOCENTER_BASE_SMILES = [
    "NC(F)C=O",   # one tetrahedral stereocentre (amino-fluoro-acetaldehyde-like)
    "OC(F)C=O",   # one tetrahedral stereocentre (fluoro-hydroxy-acetaldehyde-like)
    "CC(N)C=O",   # one tetrahedral stereocentre (2-aminopropanal-like)
    "CC(O)C=O",   # one tetrahedral stereocentre (lactaldehyde-like)
]
DOUBLEBOND_BASE_SMILES = [
    "CC=CC=O",    # one stereogenic double bond (crotonaldehyde-like, E/Z)
    "NC=CC=O",    # one stereogenic double bond (E/Z)
]


def get_stereoisomer_pair(base_smiles):
    mol0 = Chem.MolFromSmiles(base_smiles)
    if mol0 is None:
        return None
    opts = StereoEnumerationOptions(onlyUnassigned=True, unique=True)
    isomers = list(EnumerateStereoisomers(mol0, opts))
    if len(isomers) < 2:
        return None
    smi_a = Chem.MolToSmiles(isomers[0], isomericSmiles=True)
    smi_b = Chem.MolToSmiles(isomers[1], isomericSmiles=True)
    return smi_a, smi_b


def build_pair_data(smi_a, smi_b, y_placeholder=0.0):
    result_a = generate_conformers(smi_a, num_confs=1)
    result_b = generate_conformers(smi_b, num_confs=1)
    if result_a is None or result_b is None:
        return None
    mol_a, conf_ids_a = result_a
    mol_b, conf_ids_b = result_b

    data_a_stereo = mol_to_data_stereo(mol_a, conf_ids_a[0], y_placeholder)
    data_b_stereo = mol_to_data_stereo(mol_b, conf_ids_b[0], y_placeholder)
    data_a_blind = mol_to_data_blind(mol_a, conf_ids_a[0], y_placeholder)
    data_b_blind = mol_to_data_blind(mol_b, conf_ids_b[0], y_placeholder)

    if any(d is None for d in [data_a_stereo, data_b_stereo, data_a_blind, data_b_blind]):
        return None
    return data_a_stereo, data_b_stereo, data_a_blind, data_b_blind


@torch.no_grad()
def embed_molecule(encoder, data, device):
    encoder.eval()
    batch = Batch.from_data_list([data]).to(device)
    emb = encoder(batch.x, batch.pos, batch.edge_index, batch.edge_attr, batch.batch)
    return emb.cpu().numpy()[0]


def l2(u, v):
    return float(np.linalg.norm(u - v))


def cos_sim(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-8))


# Fresh, UNTRAINED stereo-blind encoder -- see markdown above for why training
# doesn't matter for this specific check.
blind_encoder_demo = EGNNEncoder(len(ATOM_TYPES), len(BOND_TYPES)).to(device)

discrimination_results = []
for base_smiles in STEREOCENTER_BASE_SMILES + DOUBLEBOND_BASE_SMILES:
    pair = get_stereoisomer_pair(base_smiles)
    if pair is None:
        print(f"Skipping {base_smiles}: could not enumerate 2 distinct stereoisomers.")
        continue
    smi_a, smi_b = pair

    built = build_pair_data(smi_a, smi_b)
    if built is None:
        print(f"Skipping {base_smiles}: conformer generation failed.")
        continue
    data_a_stereo, data_b_stereo, data_a_blind, data_b_blind = built

    emb_a_blind = embed_molecule(blind_encoder_demo, data_a_blind, device)
    emb_b_blind = embed_molecule(blind_encoder_demo, data_b_blind, device)
    emb_a_stereo = embed_molecule(model_stereo.encoder, data_a_stereo, device)
    emb_b_stereo = embed_molecule(model_stereo.encoder, data_b_stereo, device)

    row = {
        'base_smiles': base_smiles,
        'isomer_a': smi_a,
        'isomer_b': smi_b,
        'blind_embedding_l2_dist': l2(emb_a_blind, emb_b_blind),
        'blind_embedding_cos_sim': cos_sim(emb_a_blind, emb_b_blind),
        'stereo_embedding_l2_dist': l2(emb_a_stereo, emb_b_stereo),
        'stereo_embedding_cos_sim': cos_sim(emb_a_stereo, emb_b_stereo),
    }
    discrimination_results.append(row)
    print(f"{base_smiles:12s} | blind L2={row['blind_embedding_l2_dist']:.6f} "
          f"cos={row['blind_embedding_cos_sim']:.6f} || "
          f"stereo L2={row['stereo_embedding_l2_dist']:.6f} "
          f"cos={row['stereo_embedding_cos_sim']:.6f}")


## 11. Save discrimination results


In [ ]:
with open('stereoisomer_discrimination_results.json', 'w') as f:
    json.dump(discrimination_results, f, indent=2)

print(json.dumps(discrimination_results, indent=2))

if discrimination_results:
    avg_blind_l2 = np.mean([r['blind_embedding_l2_dist'] for r in discrimination_results])
    avg_stereo_l2 = np.mean([r['stereo_embedding_l2_dist'] for r in discrimination_results])
    print(f"\nAverage blind-encoder  L2 distance across pairs: {avg_blind_l2:.6f}")
    print(f"Average stereo-encoder L2 distance across pairs: {avg_stereo_l2:.6f}")


## 12. Training curve


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(history_st['train_loss'], label='Train L1 loss (normalized)')
plt.plot(history_st['val_mae_norm'], label='Val MAE (normalized)')
plt.xlabel('Epoch')
plt.ylabel('Loss / MAE')
plt.title('3D + Stereo EGNN — Training Curve')
plt.legend()
plt.show()


## Done — files to keep for later notebooks

- `results_3d_stereo_egnn.json` — QM9 gap test MAE for this model (sanity check vs Notebook 2)
- `stereoisomer_discrimination_results.json` — the R/S and E/Z embedding-discrimination table
  (good evidence for the viva: shows the encoder *can* represent stereochemical differences
  that a plain 3D EGNN provably cannot)
- `egnn_stereo_best.pt` — trained weights (optional)

**Recap of the key finding to present:** QM9 regression MAE is essentially unchanged by adding
stereo features (as expected — `gap` is achiral), but the discrimination test shows the
stereo-aware encoder assigns clearly different embeddings to molecules that are literally
indistinguishable at the 2D-graph level, while the distance-only EGNN cannot. This is exactly
the encoder we now carry forward.

**Next up (Notebook 4):** extend this same 3D + stereo-aware encoder to multi-task **Tox21**
toxicity classification (12 assay endpoints, masked BCE loss, structural-alert fingerprints for
interpretability) — the setting where stereochemistry differences can genuinely change the
predicted outcome.
